# Practical Implementation of AWQ

**Duration:** 5 min

This module covers the practical implementation of Activation-aware Weight Quantization (AWQ) for neural network model compression. AWQ is a technique that quantizes both weights and activations to reduce model size and improve inference speed without significantly compromising accuracy. Understanding and implementing AWQ is crucial for deploying efficient models in resource-constrained environments.

## Understanding AWQ

Activation-aware Weight Quantization (AWQ) is a method that quantizes the weights of a neural network based on the distribution of activations. This approach ensures that the quantized weights maintain the important characteristics of the original weights, leading to minimal loss in model performance. AWQ involves calibrating the quantization levels based on the activation statistics, which helps in preserving the model's accuracy post-quantization.

In [ ]:
import torch
import torch.nn as nn

# Define a simple neural network
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(10, 5)
        self.fc2 = nn.Linear(5, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model
model = SimpleNN()

# Function to quantize weights using AWQ
def quantize_weights(model, bits):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            # Quantize weights
            weight_quantized = torch.round(module.weight / (2**(32 - bits) - 1)) * (2**(32 - bits) - 1)
            module.weight.data = weight_quantized
    return model

# Quantize the model to 4 bits
quantized_model = quantize_weights(model, 4)

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/quantization-engineering/mod-10.ipynb)

```
Quantized model weights have been updated.
```

## Implementing AWQ in Practice

To implement AWQ in practice, one must first collect activation statistics during a calibration phase. These statistics are then used to determine the quantization levels for the weights. The quantized weights are then applied to the model, and the model is fine-tuned to adapt to the quantization. This process ensures that the quantized model performs closely to the original model.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define a simple neural network
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(10, 5)
        self.fc2 = nn.Linear(5, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model
model = SimpleNN()

# Calibration phase: Collect activation statistics
def collect_activations(model, data_loader):
    activations = []
    model.eval()
    with torch.no_grad():
        for inputs, _ in data_loader:
            outputs = model(inputs)
            activations.append(outputs.cpu().numpy())
    return np.concatenate(activations, axis=0)

# Dummy data loader
data_loader = torch.utils.data.DataLoader(torch.randn(100, 10), batch_size=10)
activation_stats = collect_activations(model, data_loader)

# Quantize weights based on activation statistics
def quantize_weights_with_stats(model, activation_stats, bits):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            # Quantize weights
            weight_quantized = torch.round(module.weight / (2**(32 - bits) - 1)) * (2**(32 - bits) - 1)
            module.weight.data = weight_quantized
    return model

# Quantize the model to 4 bits
quantized_model = quantize_weights_with_stats(model, activation_stats, 4)

# Fine-tune the quantized model
criterion = nn.MSELoss()
optimizer = optim.SGD(quantized_model.parameters(), lr=0.01)

for epoch in range(5):
    running_loss = 0.0
    for inputs, targets in data_loader:
        optimizer.zero_grad()
        outputs = quantized_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(data_loader)}') 

> **💡 Tip:** Ensure that the calibration dataset is representative of the actual data distribution to achieve effective quantization.

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary goal of AWQ?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387120512" value="0">
      <span>To increase model size</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387120512" value="1">
      <span>To reduce model size and improve inference speed</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387120512" value="2">
      <span>To increase model accuracy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387120512" value="3">
      <span>To reduce training time</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is collected during the calibration phase in AWQ?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387121152" value="0">
      <span>Model weights</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387121152" value="1">
      <span>Activation statistics</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387121152" value="2">
      <span>Loss values</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387121152" value="3">
      <span>Gradient values</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>